# Notebook 1: The Dual-Write Problem

Many services do two things in a single request handler:

1. **Write to the database** (e.g. `INSERT INTO orders ...`)
2. **Publish an event** to a message bus (Kafka, RabbitMQ, SQS) so other services know about it.

These are **two different systems**. They do not share a transaction. Whatever order you do them in, a crash or network blip between them leaves the system in a broken state. That broken state is usually **silent** — no error, no alert — just missing events or phantom events.

This notebook shows the problem from three angles so the fix in notebook 2 makes sense.

## Setup

Start Postgres and Adminer (browse the DB at http://localhost:8080):

```bash
cd 04-patterns/outbox-and-cdc
docker compose up -d
uv sync
```

Adminer login: server=`postgres`, user=`demo`, password=`demo`, database=`outbox_demo`.

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window (`Cmd+Shift+P` -> **Reload Window**).

## BAD attempt #1: write DB, then publish

The handler commits the order to Postgres, then tries to publish. We simulate a crash (process killed, network blip, Kafka down) between the two steps.

In [ ]:
import psycopg
DSN = 'host=localhost port=5432 user=demo password=demo dbname=outbox_demo'

with psycopg.connect(DSN, autocommit=True) as conn:
    conn.execute('DROP TABLE IF EXISTS orders')
    conn.execute('CREATE TABLE orders (id SERIAL PRIMARY KEY, item TEXT, total INTEGER)')

published = []  # pretend this list is a Kafka topic

def place_order_db_first(item, total, crash_after_db=False):
    with psycopg.connect(DSN) as conn:
        with conn.transaction():
            cur = conn.execute(
                'INSERT INTO orders(item,total) VALUES (%s,%s) RETURNING id',
                (item, total),
            )
            order_id = cur.fetchone()[0]
        # transaction committed here
        if crash_after_db:
            raise RuntimeError('crashed after DB commit, before publish')
        published.append({'order_id': order_id, 'item': item, 'total': total})
        return order_id

try:
    place_order_db_first('book', 25, crash_after_db=True)
except RuntimeError as e:
    print(e)

with psycopg.connect(DSN) as conn:
    rows = conn.execute('SELECT * FROM orders').fetchall()
print('orders in DB    :', rows)
print('events published:', published)
print('>> the DB has the order, but no event was published - downstream will never know')

## BAD attempt #2: publish first, then write the DB

A common *intuitive* fix is to flip the order: publish first, then write to the DB. Now we get the opposite bug - a **phantom event** for an order that was never actually saved.

In [ ]:
with psycopg.connect(DSN, autocommit=True) as conn:
    conn.execute('TRUNCATE orders RESTART IDENTITY')
published.clear()

def place_order_publish_first(item, total, crash_after_publish=False):
    # publish first
    published.append({'item': item, 'total': total})
    if crash_after_publish:
        raise RuntimeError('crashed after publish, before DB write')
    with psycopg.connect(DSN) as conn:
        with conn.transaction():
            conn.execute('INSERT INTO orders(item,total) VALUES (%s,%s)', (item, total))

try:
    place_order_publish_first('lamp', 40, crash_after_publish=True)
except RuntimeError as e:
    print(e)

with psycopg.connect(DSN) as conn:
    print('orders in DB    :', conn.execute('SELECT * FROM orders').fetchall())
print('events published:', published)
print('>> the bus has a phantom event for an order that was never saved')

## BAD attempt #3: just retry the publish

The next instinct is: keep retrying the publish until it succeeds. That helps with transient failures, but it does **not** fix the fundamental problem:

- The process itself can die (OOM, pod evicted, power loss) *after* the DB commit but *before* the retry loop ever starts.
- Even a bounded retry eventually gives up.
- A long retry loop holds the request open, coupling response latency to the bus's availability.

The root cause is simple: **two systems, no shared transaction**. No amount of ordering or retrying inside one handler can make two independent commits atomic.

## Why not just use two-phase commit (2PC / XA)?

It looks like a perfect fit: wrap the database and the message bus in one *distributed* transaction so either both commit or neither does. In practice, teams avoid 2PC because:

- Most modern brokers (Kafka, Kinesis, SQS, NATS) **don't speak XA** at all.
- 2PC **blocks** when any participant is slow or crashed - it trades availability for atomicity, which is usually the opposite of why you added a message bus.
- Running a heterogeneous transaction manager across services is operationally hard and error-prone.

The transactional outbox in notebook 2 gives you the same guarantee (data + event are atomic) using only the one transactional resource you already have: your database.


## Where this bites in production

- **Orders service** commits an order but the `order.placed` event is lost - the warehouse never ships it, and the customer gets silence.
- **Payments service** charges the card (external API) but crashes before the DB write - the customer is charged for an order that does not exist.
- **User signup** writes the user row but the `user.created` event to analytics / welcome-email is dropped - new users never get onboarded.

These bugs are **silent and rare**, which is the worst kind. They surface days later as ghost data and confused customers.

Next notebook: write the event **inside the same transaction** as the data, into an `outbox` table. A separate publisher reads that table and ships the events. One commit, two guarantees.